# EduHinglish: Gemma-3-4B-it QLoRA — v4, all subjects, resumable (T4)

Trains the Hinglish answer generator on **Biology, Chemistry, Physics, Maths and Social Science (Class 9 & 10)**.

**Resumable:** checkpoints go to Google Drive every `SAVE_STEPS` steps. When the Colab runtime disconnects,
reconnect and **Run all** again — training continues from the last checkpoint (same step, optimizer,
LR schedule and data order), it does not start over. Once training is finished, re-running skips
straight to the saved adapter.

> **Runtime**: Runtime → Change runtime type → **T4 GPU**

## 0. Install dependencies
Same pins as the v3 run (notebook 04). **Restart the runtime once after the first install** (Runtime → Restart session), then Run all.

In [ ]:
!pip install -q -U \
    "bitsandbytes>=0.43.0" \
    "transformers>=4.47.0" \
    "peft>=0.14.0" \
    "accelerate>=1.0.0" \
    "datasets>=3.0.0" \
    "trl>=0.12.0,<0.20.0" \
    "huggingface_hub" \
    "kagglehub"

## 1. Settings — the only cell you normally edit

In [ ]:
BASE_DIR = "/content/drive/MyDrive/EduHinglish"
RUN_NAME = "gemma_eduhinglish_v4"

# Where the 5 dataset JSONs come from:
#   "drive"  -> {BASE_DIR}/data/<file>   (upload them from the repo's data/ folder once)
#   "kaggle" -> the Kaggle dataset published with scripts/publish_kaggle_dataset.py
DATA_SOURCE    = "drive"
KAGGLE_DATASET = "your-kaggle-username/eduhinglish-ncert-hinglish-qa"   # only for DATA_SOURCE="kaggle"
DATA_FILES = ["bio_9_10.json", "chem_9_10.json", "phys_9_10.json", "maths_9_10.json", "ss_9_10.json"]

# Start the NEW run from the v3 adapter (bio + ss) instead of from scratch. Only used when no
# v4 checkpoint exists yet; set to None to train from the plain base model.
WARM_START_ADAPTER = f"{BASE_DIR}/models/gemma_eduhinglish_v3/adapter"

NUM_EPOCHS = 2        # ~3,400 train rows / effective batch 8 ≈ 430 steps per epoch (~50 min/epoch on T4)
SAVE_STEPS = 50       # checkpoint every ~6 min, so a disconnect loses at most a few minutes
EVAL_STEPS = 100

# Hugging Face Hub (needs a WRITE token in the HF_TOKEN secret)
HF_REPO_ID = "your-hf-username/eduhinglish-gemma-3-4b-it-lora"
PUSH_FINAL_ADAPTER_TO_HUB = True
# Also push every checkpoint to the Hub (folder "last-checkpoint"). Lets you resume even if Drive
# is unavailable, at the cost of a ~400 MB upload per save. Drive alone is usually enough.
PUSH_CHECKPOINTS_TO_HUB = False

CKPT_DIR    = f"{BASE_DIR}/models/{RUN_NAME}/checkpoints"
ADAPTER_DIR = f"{BASE_DIR}/models/{RUN_NAME}/adapter"

## 2. Authentication & Drive
Gemma is gated: accept the licence at https://huggingface.co/google/gemma-3-4b-it.
Add a **Write** token from https://huggingface.co/settings/tokens to Colab **Secrets** as `HF_TOKEN` (enable notebook access).
For `DATA_SOURCE="kaggle"` also add `KAGGLE_USERNAME` and `KAGGLE_KEY` secrets.

In [ ]:
import os
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
os.makedirs(CKPT_DIR, exist_ok=True)

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])

if DATA_SOURCE == "kaggle":
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
print("Logged in.")

## 3. Training state — what has been done so far

In [ ]:
import json, glob
from transformers.trainer_utils import get_last_checkpoint

def find_last_checkpoint():
    last = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
    if last is None and PUSH_CHECKPOINTS_TO_HUB:
        # Drive copy missing: try the checkpoint pushed to the Hub
        from huggingface_hub import snapshot_download
        try:
            path = snapshot_download(HF_REPO_ID, allow_patterns=["last-checkpoint/*"],
                                     local_dir=f"{CKPT_DIR}/_hub")
            if os.path.exists(f"{path}/last-checkpoint/trainer_state.json"):
                last = f"{path}/last-checkpoint"
        except Exception as e:
            print("No Hub checkpoint:", e)
    return last

LAST_CKPT = find_last_checkpoint()
TRAINING_DONE = os.path.exists(f"{ADAPTER_DIR}/adapter_config.json") and \
                os.path.exists(f"{BASE_DIR}/models/{RUN_NAME}/TRAINING_DONE")

if TRAINING_DONE:
    print(f"Training already finished. Final adapter: {ADAPTER_DIR}")
elif LAST_CKPT:
    st = json.load(open(f"{LAST_CKPT}/trainer_state.json"))
    print(f"Resuming from {LAST_CKPT}")
    print(f"  step {st['global_step']} / {st['max_steps']}  (epoch {st['epoch']:.2f})")
    losses = [h for h in st["log_history"] if "loss" in h]
    evals  = [h for h in st["log_history"] if "eval_loss" in h]
    if losses: print(f"  last train loss {losses[-1]['loss']:.4f}")
    if evals:  print(f"  last eval  loss {evals[-1]['eval_loss']:.4f}")
else:
    print("No checkpoint yet — starting a new run.")

## 4. Load the dataset (all five subjects)
Each record already has a `text` field rendered by `scripts/build_dataset.py` in the exact prompt
format `src/generator.py` uses at inference (context, question, optional intent line, instruction),
so the notebook does no prompt formatting of its own.

The train/eval split is decided by a hash of each record's `id`, so it is identical on every
session (needed for a correct resume) and stays stable when new chapters are added.

In [ ]:
import hashlib
from collections import Counter
from datasets import Dataset

if DATA_SOURCE == "kaggle":
    import kagglehub
    data_dir = kagglehub.dataset_download(KAGGLE_DATASET)
else:
    data_dir = f"{BASE_DIR}/data"

rows = []
for name in DATA_FILES:
    with open(f"{data_dir}/{name}", encoding="utf-8") as f:
        entries = json.load(f)
    print(f"{name:18s} {len(entries):5d} entries")
    rows.extend(entries)

def is_eval(rid, share=0.05):
    return int(hashlib.md5(rid.encode()).hexdigest(), 16) % 10_000 < share * 10_000

train_rows = [r for r in rows if not is_eval(r["id"])]
eval_rows  = [r for r in rows if is_eval(r["id"])]
print(f"\nTotal {len(rows)} | train {len(train_rows)} | eval {len(eval_rows)}")
print("By subject:", dict(Counter(r["subject"] for r in rows)))

train_dataset = Dataset.from_list([{"text": r["text"]} for r in train_rows]).shuffle(seed=42)
eval_dataset  = Dataset.from_list([{"text": r["text"]} for r in eval_rows])
print("\nSample:\n", train_dataset[0]["text"][:800])

## 5. Model + LoRA (QLoRA, fp16 — T4 has no bf16)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model

MODEL_ID = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
model.config.use_cache = False
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

if TRAINING_DONE:
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    print("Loaded finished adapter (no training needed).")
elif LAST_CKPT:
    # Fresh LoRA layers; trainer.train(resume_from_checkpoint=...) loads the checkpoint weights into them
    model = get_peft_model(model, lora_config)
    print("LoRA initialised — weights will be restored from", LAST_CKPT)
elif WARM_START_ADAPTER and os.path.exists(f"{WARM_START_ADAPTER}/adapter_config.json"):
    model = PeftModel.from_pretrained(model, WARM_START_ADAPTER, is_trainable=True)
    print("Warm start from", WARM_START_ADAPTER)
else:
    model = get_peft_model(model, lora_config)
    print("Training LoRA from scratch.")
model.print_trainable_parameters()

## 6. Train (or resume)
Safe to re-run at any time. If the runtime dies mid-training, reconnect and **Run all**.

In [ ]:
import math
from trl import SFTTrainer, SFTConfig

total_steps = math.ceil(len(train_dataset) / (2 * 4)) * NUM_EPOCHS
print(f"Planned optimizer steps: {total_steps}")

sft_config = SFTConfig(
    output_dir=CKPT_DIR,
    dataset_text_field="text",
    max_length=1024,
    packing=False,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    warmup_steps=max(1, int(0.03 * total_steps)),
    lr_scheduler_type="cosine",
    fp16=True,
    bf16=False,

    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,          # keep the 2 newest checkpoints on Drive
    seed=42,
    data_seed=42,
    report_to="none",

    push_to_hub=PUSH_CHECKPOINTS_TO_HUB,
    hub_model_id=HF_REPO_ID if PUSH_CHECKPOINTS_TO_HUB else None,
    hub_strategy="checkpoint",
    hub_private_repo=True,
)

if TRAINING_DONE:
    trainer = None
    print("Already trained — skipping.")
else:
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        args=sft_config,
    )
    trainer.train(resume_from_checkpoint=LAST_CKPT)   # None = new run

## 7. Save the final adapter to Drive

In [ ]:
if not TRAINING_DONE:
    trainer.model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    metrics = trainer.evaluate()
    with open(f"{ADAPTER_DIR}/eval_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    open(f"{BASE_DIR}/models/{RUN_NAME}/TRAINING_DONE", "w").write(json.dumps(metrics))
    TRAINING_DONE = True
    print("Saved", ADAPTER_DIR, metrics)
else:
    print("Adapter already at", ADAPTER_DIR)

## 8. Push the adapter to the Hugging Face Hub
Uploads only the LoRA adapter (~130 MB). Anyone with access loads it on top of `google/gemma-3-4b-it`.

In [ ]:
from huggingface_hub import HfApi

CARD = f'''---
base_model: google/gemma-3-4b-it
library_name: peft
license: gemma
language: [en, hi]
tags: [lora, qlora, hinglish, education, ncert, code-mixing]
pipeline_tag: text-generation
---
# EduHinglish — Gemma-3-4B-it LoRA ({RUN_NAME})

LoRA adapter that answers NCERT Class 9–10 questions (Biology, Chemistry, Physics, Mathematics,
Social Science) in Roman-script Hinglish, grounded in a retrieved textbook passage.

Trained with QLoRA (4-bit NF4, r=16, alpha=32) on {len(train_rows)} curated QA pairs
({len(eval_rows)} held out). Eval metrics: see `eval_metrics.json`.

## Prompt format (single user turn)
```
Context:
<NCERT passage>

Question:
<student question>

Intent Type: <intent>          # optional line
Answer the question in Hinglish using only the given context.
```

## Usage
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
tok = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")
base = AutoModelForCausalLM.from_pretrained("google/gemma-3-4b-it", device_map="auto")
model = PeftModel.from_pretrained(base, "{HF_REPO_ID}")
```
Use of this adapter is subject to the Gemma Terms of Use.
'''

if PUSH_FINAL_ADAPTER_TO_HUB:
    with open(f"{ADAPTER_DIR}/README.md", "w") as f:
        f.write(CARD)
    api = HfApi()
    api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
    api.upload_folder(repo_id=HF_REPO_ID, folder_path=ADAPTER_DIR,
                      commit_message=f"{RUN_NAME} adapter")
    print(f"Pushed: https://huggingface.co/{HF_REPO_ID}")

## 9. Quick check — one question per subject + held-out examples
Uses the same prompt builder as `src/generator.py`.

In [ ]:
model = trainer.model if trainer is not None else model
model.eval()
model.config.use_cache = True
END_TURN = tokenizer.convert_tokens_to_ids("<end_of_turn>")

def build_prompt(context, question, intent=None, language="hinglish"):
    intent_line = f"Intent Type: {intent}\n" if intent else ""
    instruction = f"Answer the question in {'Hinglish' if language == 'hinglish' else 'English'} using only the given context."
    return f"Context:\n{context}\n\nQuestion:\n{question}\n\n{intent_line}{instruction}"

@torch.no_grad()
def ask(context, question, intent=None, max_new_tokens=256):
    text = tokenizer.apply_chat_template([{"role": "user", "content": build_prompt(context, question, intent)}],
                                         tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                         pad_token_id=tokenizer.pad_token_id, eos_token_id=[tokenizer.eos_token_id, END_TURN])
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

by_subject = {}
for r in eval_rows:
    by_subject.setdefault(r["subject"], r)
for subject, r in by_subject.items():
    print(f"=== {subject} | {r['chapter']} [{r['intent']}]")
    print("Q:  ", r["question_hinglish"])
    print("GEN:", ask(r["ncert_context"], r["question_hinglish"], r["intent"]))
    print("REF:", r["answer_hinglish"], "\n")